# GSC 2026 Quick Start

Run this notebook from the extracted `challenge_starter/` directory.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

ROOT = Path.cwd().resolve()

if not (ROOT / "model.py").is_file():
    raise FileNotFoundError(
        "Open this notebook from the challenge_starter directory."
    )

print("Starter root:", ROOT)

## Inspect separate Attack Case 1 model files

In [ ]:
from utilities.model_io import load_state_dict_directory

case_1_models = load_state_dict_directory(
    ROOT / "attack" / "case_1",
    expected_count=8,
)

print("Attack Case 1 models:", len(case_1_models))

for name, tensor in case_1_models[0].items():
    print(
        f"{name:24s} "
        f"shape={tuple(tensor.shape)} "
        f"norm={tensor.norm().item():.4f}"
    )

## Create a non-competitive attack baseline

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(ROOT / "attack" / "attack_baseline.py"),
        "--output-root",
        str(ROOT / "participant_models"),
    ],
    check=True,
)

## Generate and validate the attack CSV

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(ROOT / "attack" / "create_attack_submission.py"),
        "--models-root",
        str(ROOT / "participant_models"),
        "--output",
        str(ROOT / "attack_submission.csv"),
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        str(ROOT / "attack" / "validate_attack_submission.py"),
        "--submission",
        str(ROOT / "attack_submission.csv"),
    ],
    check=True,
)

## Create and test the defense submission

In [ ]:
defense_submission = ROOT / "defense_submission.py"

if not defense_submission.exists():
    shutil.copy2(
        ROOT / "defense" / "defense_submission_template.py",
        defense_submission,
    )

subprocess.run(
    [
        sys.executable,
        str(ROOT / "defense" / "test_defense_submission.py"),
        "--submission",
        str(defense_submission),
        "--visible-case-dir",
        str(ROOT / "defense" / "visible_case"),
    ],
    check=True,
)

## Final files

Upload `attack_submission.csv` and `defense_submission.py`.